# Li₂S at 24 qubits — HI-VQE on Qiskit

**Handover Iterative Variational Quantum Eigensolver** ([arXiv:2503.06292](https://arxiv.org/abs/2503.06292),
Qunova Computing) on that paper's own 24-qubit benchmark: **Li₂S, CAS(12e,12o)/STO-3G,
853,776 determinants**, along the Li–S bond dissociation coordinate.

The method asks the device a different question than a conventional VQE does. Instead
of *what is the energy* — 15,697 Pauli words to measure, every iteration — it asks
**which electron configurations matter**, then projects the Hamiltonian onto the
subspace those configurations span and diagonalizes it exactly, classically. That is
the handover, and it is why one measurement setting per iteration replaces 15,697.

Everything below runs on a free Colab **CPU** runtime. No GPU, no QPU, no account
anywhere — the whole study, quantum chemistry included, fits in one session.

**Open this notebook straight from GitHub.** The repository is public, so this link
needs no account, no token and no authorization step:

```
https://colab.research.google.com/github/bhargav2603/qubit_run/blob/main/studies/li2s-24q/colab.ipynb
```

Uploading the `.ipynb` to Colab by hand (File → Upload notebook) works just as well.
Either way, §3 fetches the code.

| § | Step | Roughly |
|---|---|---|
| 1 | Environment check | seconds |
| 2 | Install the stack | 1–3 min |
| 3 | Clone the repo | seconds |
| 4 | Self-test — prove the stack before spending compute | ~30 s |
| 5 | The orbital window, before spending an SCF on it | ~1 min |
| 6 | `prepare` + `validate` at equilibrium — writes the receipt | 1–3 min |
| 7 | `hivqe` — the published method, one geometry | 4–8 min |
| 8 | The ablation — what the quantum layer actually contributed | 5–10 min |
| 9 | **The full 13-point dissociation scan** | **15–25 min, + `prepare`** |
| 10 | Classical baselines and the measurement-cost count | 2–5 min |
| 11 | **The seven figures** | ~1 min |
| 12 | The assembled report | ~1 min |
| 13 | Download everything | seconds |

**Single point only: about 10 minutes.** Skip §9 and you still have a validated
result, an ablation, every figure that does not need the curve, and a full report.

**In a hurry on the scan?** Append `--optimizer none --workers 2` to §9's `scan`
call, which roughly halves it again: sampling dominates the wall time, SPSA asks for
three circuit evaluations per iteration instead of one, and the geometries are
independent so they parallelize cleanly. On the benchmark measured for this folder it
changed the energy by nothing at all.

To run fewer points, set `SCAN_DISTANCES` in §9 to any subset — but keep at least one
geometry either side of the 2.10 Å minimum, or §12 declines to report a dissociation
energy (correctly: an unbracketed minimum has no well to measure).

**The qubit count does not move.** 24 qubits is fixed by CAS(12e,12o) — 12 spatial
orbitals under Jordan–Wigner — and is a property of the active space, not of how many
geometries the scan visits or how many HI-VQE iterations each one runs.

Do **not** reach for `--max-determinants 6000` to go faster. Measured, it saves
nothing — the classical work per iteration is nearly flat between 6,000 and 20,000
determinants — and it cost 170 mHa of accuracy.

Runtime → Run all works, but read the output of §4 and §6 before trusting anything
downstream: both are gates, not formalities.

> **Scope, stated up front.** At 24 qubits the exact answer is classically cheap —
> 853,776 determinants is a direct diagonalization. **Nothing here is evidence of
> quantum advantage.** The ablation in §8 is included precisely so the quantum
> layer's actual contribution is a number you read rather than a claim you accept.

## 1. Environment check

Nothing here is Colab-specific — the same cells work in any Linux/macOS shell with
the `!` prefixes dropped.

**Windows is not fatal here, unlike in most quantum-chemistry notebooks.** PySCF
publishes no Windows wheel, but it is needed by exactly three commands — `prepare`,
`geometry` and `classical`. Everything else reads a JSON cache. So the honest
Windows workflow is: run §6 and §10 here once, download the cache in §13, and every
remaining command runs natively on the Windows machine forever after.

In [ ]:
import platform, sys

print(f"Python   : {platform.python_version()}")
print(f"Platform : {platform.system()} {platform.machine()}")

major, minor = sys.version_info[:2]
if (major, minor) < (3, 10):
    print("\nWARNING: this workflow is developed on 3.12 and needs at least 3.10.")
elif (major, minor) >= (3, 14):
    print("\nWARNING: newer than the tested 3.12; wheels may be missing.")
else:
    print("\nVersion is fine.")

if platform.system() == "Windows":
    print(
        "\nWindows: PySCF has no wheel here, so `prepare`, `geometry` and "
        "`classical` will fail.\nEverything else -- selftest, validate, hivqe, "
        "scan, plot, report -- runs fine on a\ndownloaded cache. That is the "
        "whole reason this notebook exists."
    )

## 2. Install

`pyscf` is the one that matters and the one Windows cannot have. `qiskit` and
`qiskit-aer` build and run the sampling circuit; `matplotlib` is figures only —
nothing in the physics or simulator path imports it. The version bounds are exactly
those in the repository's `requirements.txt`, so a Colab session and a local checkout
resolve to the same stack.

**`openfermion` installs on its own line, deliberately.** It is used in two places
only — the self-test's cross-check against an independent Jordan–Wigner
implementation, and the Pauli-word count in §10 — and both skip cleanly with a
printed message when it is absent. It also pulls in `cirq-core`, which is the one
dependency here that can genuinely fail to resolve on a given Colab image. Keeping it
off the first line means such a failure costs those two checks and nothing else,
rather than taking `pyscf` and `qiskit` down with it.

**numpy and scipy are deliberately not listed.** Colab already ships versions that
satisfy everything here, and naming them explicitly makes pip resolve them fresh to
the newest release — which upgrades numpy out from under Colab's preinstalled
`numba` and drags `pandas` along with it, producing a wall of red dependency-conflict
text. Letting the dependencies ask for what they need keeps Colab's own packages
intact.

If you *do* see conflicts mentioning `pandas`, `numba` or `google-colab`, they are
about Colab's preinstalled packages, not this workflow — nothing here imports any of
them. The cell after the install verifies what actually matters: that the stack
imports and reports its versions. **That, not pip's resolver, is the test.**

In [ ]:
# The stack the physics actually needs. Bounds match the repo's requirements.txt.
!pip install -q "pyscf>=2.5" "qiskit>=1.2,<3" "qiskit-aer>=0.15" "matplotlib>=3.8"

# The optional cross-check, on its own line so a failure here is not fatal.
!pip install -q "openfermion>=1.6,<2" || echo "openfermion unavailable -- the self-test will skip its independent Jordan-Wigner check and 'run.py paulis' in section 10 will not run. Nothing else is affected."


In [ ]:
# Does the stack actually import and work? pip's resolver warnings do not
# answer that; this does.
import importlib
from importlib import metadata

REQUIRED = ["numpy", "scipy", "pyscf", "qiskit", "qiskit_aer", "openfermion", "matplotlib"]
DISTRIBUTION = {"qiskit_aer": "qiskit-aer"}

missing, incompatible = [], []
for module in REQUIRED:
    try:
        importlib.import_module(module)
        print(f"  ok        {module:<14} {metadata.version(DISTRIBUTION.get(module, module))}")
    except ModuleNotFoundError as error:
        # Simply not installed. Nothing subtle about it.
        missing.append(module)
        print(f"  MISSING   {module:<14} {error}")
    except Exception as error:
        # Installed but will not load. At this point in a Colab session that
        # almost always means numpy was replaced underneath a C extension
        # compiled against the previous ABI -- which no amount of reinstalling
        # fixes, because the stale module is already in memory.
        incompatible.append(module)
        print(f"  BROKEN    {module:<14} {type(error).__name__}: {error}")

print()
if incompatible:
    print("=" * 70)
    print("Runtime -> Restart session, then re-run THIS cell only (not the")
    print("install cell). A binary extension is holding a numpy that has since")
    print("been replaced; only a restart clears it.")
    print("=" * 70)
elif missing:
    print("=" * 70)
    print(f"Not installed: {', '.join(missing)}")
    print("Re-run the install cell above. If pyscf is the only one missing and")
    print("you are NOT on Colab, note that it has no Windows wheel -- that is")
    print("expected, and only affects prepare / geometry / classical.")
    print("=" * 70)
else:
    import numpy, scipy
    print(f"All {len(REQUIRED)} imports fine. numpy {numpy.__version__} / scipy {scipy.__version__}")
    print()
    print("Dependency-conflict warnings naming pandas, numba or google-colab are")
    print("about Colab's own preinstalled packages and do not affect this")
    print("workflow -- nothing in it imports any of them. This table is the test.")

## 3. Get the workflow

**`bhargav2603/qubit_run` is a public repository**, so the cell below is a plain
anonymous `git clone` — no personal access token, no `getpass` prompt, no GitHub
account, and nothing left on disk that has to be scrubbed afterwards.

The workflow lives in the `studies/li2s-24q` folder of that repository. The cell locates
`run.py` rather than assuming the layout, `chdir`s into its folder, and puts that
folder on `sys.path` so `import visualize` works in §11. It then lists any module it
expected and did not find, which is the fastest way to catch a partial clone.

If this fails it is a network or a naming problem, not a permissions one. The next
cell is the fallback for that case: upload the folder by hand, no network required.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

OWNER, NAME = "bhargav2603", "qubit_run"
BRANCH = "main"
FOLDER = "studies/li2s-24q"        # the workflow folder inside the repo

URL = f"https://github.com/{OWNER}/{NAME}.git"
target = Path("/content/repo")

if target.exists():
    shutil.rmtree(target)          # always start from a clean clone

finished = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, URL, str(target)],
    capture_output=True,
    text=True,
)
if finished.returncode != 0:
    message = finished.stderr.strip() or f"git exited {finished.returncode}"
    raise SystemExit(
        f"Could not clone {OWNER}/{NAME}.\n  {message.splitlines()[-1]}\n\n"
        "The repository is public, so this is not a credentials problem -- check\n"
        "the network, or that the repository has not been renamed. Failing that,\n"
        "skip this cell and use the upload fallback below, which needs no network."
    )

# Find run.py rather than assuming the layout, so a renamed or nested folder
# still works.
candidates = sorted(target.rglob("run.py"))
matching = [p for p in candidates if p.parent.name == FOLDER] or candidates
if not matching:
    raise SystemExit(f"Cloned {OWNER}/{NAME}, but it contains no run.py")
ROOT = matching[0].parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))   # so `import visualize` works in section 11

commit = subprocess.run(
    ["git", "-C", str(target), "log", "-1", "--format=%h %ad %s", "--date=short"],
    capture_output=True, text=True,
).stdout.strip()
print(f"Working in : {ROOT}")
print(f"Commit     : {commit}\n")

required = [
    "molecule.py", "hamiltonian.py", "determinants.py", "ansatz.py",
    "sector_sim.py", "hivqe.py", "driver.py", "validation.py",
    "classical.py", "selftest.py", "summary.py", "visualize.py", "report.py",
]
missing = [name for name in required if not (ROOT / name).is_file()]
print("Missing:", missing if missing else "nothing")


In [ ]:
# FALLBACK -- skip this cell if the clone above worked.
#
# The no-network route. Drag the studies/li2s-24q folder (or a zip of it) into Colab's
# file browser on the left, then uncomment everything below and run this cell. It
# unpacks any zip it finds under /content, then searches for run.py.
#
#   PowerShell:  Compress-Archive -Path studies/li2s-24q -DestinationPath wf.zip -Force
#
# import os, sys, zipfile
# from pathlib import Path
#
# for archive in Path("/content").glob("*.zip"):
#     with zipfile.ZipFile(archive) as handle:
#         handle.extractall("/content/workflow")
#     print("unpacked", archive.name)
#
# found = (sorted(Path("/content").rglob("studies/li2s-24q/run.py"))
#          or sorted(Path("/content").rglob("run.py")))
# if not found:
#     raise SystemExit("No run.py anywhere under /content -- did you upload the folder?")
# os.chdir(found[0].parent)
# sys.path.insert(0, str(found[0].parent))
# print("Working in", found[0].parent)

## 4. Prove the stack before spending compute on chemistry

This needs no PySCF and no cache. It builds a structurally real active space out of
random-but-valid integrals and checks:

* **module layering** — that `validate` never imports Qiskit and only
  `hamiltonian.py` imports PySCF, which is what lets the two halves of this workflow
  live on different machines;
* the fermionic phase conventions and the whole determinant engine against
  **OpenFermion's independent Jordan–Wigner operator** — a package that shares no
  code with this one;
* the sector simulator against Qiskit Aer, amplitude by amplitude;
* a complete HI-VQE run against an exactly diagonalized system.

**If this is not green, stop.** Two of these checks exist because they caught real
bugs, and both were the kind that produce a plausible wrong number rather than an
error: a global sign on one of five Slater–Condon cases, and a projection that
computed `P E P E P` instead of `P (E E) P` and so returned energies *below* the
exact ground state.

`backend` additionally reports the Qiskit/Aer environment; add `--full` to
cross-check both simulators on the real 24-qubit circuit, which is slower.

In [ ]:
!python run.py selftest
!python -m unittest discover -s tests 2>&1 | tail -3
!python run.py backend

## 5. Look at the orbital window before spending an SCF on it

`--diagnose` runs RHF once and prints every orbital with its energy and its Mulliken
population on each atom, and builds nothing.

This matters more than it sounds. Li₂S has **22 electrons in 19 STO-3G orbitals**, so
CAS(12e,12o) fixes everything else by arithmetic: 5 frozen orbitals, 2 dropped
virtuals, 12 spatial orbitals → **24 qubits**, C(12,6)² = **853,776 determinants**,
matching the paper's Table 1 exactly.

Those 5 frozen orbitals *should* be the sulfur 1s/2s/2p shell — near −91, −9 and
−6.7 Ha, while nothing else in the molecule is below −3 Ha. `prepare` **measures**
that rather than assuming it, and refuses to build if the localization on sulfur or
the core/active gap falls short.

In [ ]:
!python run.py prepare --diagnose

## 6. Build the Hamiltonian, then prove it correct — this is what writes the receipt

`validate` checks the integral symmetries, that our Slater–Condon energy of the
Hartree–Fock determinant equals PySCF's RHF energy exactly, that the projected
Hamiltonian equals Slater–Condon element by element, and that no random subspace
falls below the CASCI reference. It then writes a hash-bound receipt, and **`hivqe`
refuses to run without one.**

The receipt is bound to the physics only — the cache, the molecule specification, and
`molecule.py` / `hamiltonian.py` / `determinants.py` / `validation.py`. Editing
`ansatz.py` or `visualize.py` does not invalidate it, which is the point on a machine
that cannot reinstall PySCF to earn a new one.

2.10 Å is this folder's own CASCI/STO-3G symmetric-stretch minimum, not an
experimental number — `python run.py geometry` recomputes it. An STO-3G study should
be measured against an STO-3G minimum.

Do not continue unless this passes.

In [ ]:
!python run.py prepare  --distance 2.10
!python run.py validate --distance 2.10

## 7. HI-VQE — the published method

Each iteration: sample the Qiskit circuit for electron configurations, add them to
the accumulated subspace, project **H** into it and diagonalize exactly, drop
configurations with no amplitude, add the best single and double excitations of the
leading ones, and move θ.

The ansatz is a hardware-efficient unitary cluster Jastrow — nearest-neighbour Givens
brickworks (`XXPlusYY`) alternating with diagonal number–number layers (`RZ`, `RZZ`).
Every gate commutes with the α and β number operators separately, so on a noiseless
simulator **every shot is a valid configuration**: leakage is exactly zero rather
than merely small, and no shot is wasted on electron-count filtering.

Read three things in the output:

* **the error against CASCI** — the benchmark. 1.6 mHa is the pass mark.
* **the verdict line** — `CHEMICAL ACCURACY`, `OUTSIDE CHEMICAL ACCURACY` (raise
  `--max-determinants` or `--expansion`), `SPIN CONTAMINATED` (the subspace is not
  spin-complete), or `INCONSISTENT` — which means the energy came out *below* CASCI.
  That is variationally impossible for a projected subspace, so it is a bug report,
  not a result: re-run `validate` and `selftest`, and do not report the number.
* **`subspace_fraction`** — how little of the 853,776-determinant space it needed.
  This is one of the three quantities that genuinely *is* comparable with the paper.
* **`+ PT2 correction` and `energy + PT2`** — the Epstein–Nesbet second-order energy
  from the determinants left *outside* the subspace. It is reported beside the
  variational energy and never folded into it: PT2 is not variational, so the strict
  upper bound applies to the line above it. On the systems this was measured on it
  improved the error in 18 runs out of 18. If the run instead prints
  `PT2 IS NOT USABLE HERE`, the correction came out larger than half the correlation
  energy the subspace captured — perturbation theory is outside its domain there and
  the number should be ignored, not quoted.

The total energy is **not** comparable with theirs. The paper publishes no geometry,
no per-point energies and no orbital set, so total energies are not reproducible from
it and this folder does not claim to reproduce them.

In [ ]:
!python run.py hivqe --distance 2.10 --max-determinants 20000 --max-iterations 12

Three settings worth knowing, all of which change the answer rather than the cost.
Each default was picked by measuring 18 runs on systems whose exact energy is known,
and the README records the numbers.

| Flag | What it does | When to reach for it |
|---|---|---|
| `--ranking coupling` | ranks expansion candidates by the bare coupling instead of the Epstein–Nesbet energy gain | to reproduce the paper's selection rule exactly, with `--expansion-references 1` |
| `--spin-complete` | forces the α and β string sets to be one shared set, so the subspace is closed under the spin flip | when a run comes back `SPIN CONTAMINATED`. It buys that by spending dimension, so it costs a little energy — off by default for exactly that reason |
| `--no-pt2` | skips the perturbative correction | almost never; it costs one extra sigma product |

`--optimizer none` is worth repeating here for a different reason than speed: with
θ fixed the sector state is built **once** for the whole run rather than once per
iteration (3.60 s, then 0.036 s per repeat at 24 qubits), which is about 39 seconds
off every geometry.

In [ ]:
# The paper's own selection rule, for comparison against the default above.
!python run.py hivqe --distance 2.10 --max-determinants 20000 --max-iterations 12 --ranking coupling --expansion-references 1 --tag paper-rule

## 8. The ablation — the honest part of the study

Two runs that each remove one half of the method:

* **`--simulator none`** removes the quantum layer entirely, leaving a purely
  classical selected CI. This is the control that decides whether the sampler earned
  its place.
* **`--no-expansion`** removes the classical half instead, leaving only what the
  circuit proposes.

Compare all three in the report's ablation table, and especially in the
**first-iteration** column — a sampler that proposes good configurations shows up
there, before the classical expansion has had a chance to find them anyway.

At this size the classical control is strong. That is expected, it is the point of
benchmarking where the answer is already known, and reporting it is what makes the
rest of the numbers worth anything.

In [ ]:
!python run.py hivqe --distance 2.10 --max-determinants 20000 --max-iterations 12 --simulator none
!python run.py hivqe --distance 2.10 --max-determinants 20000 --max-iterations 12 --no-expansion

## 9. The full dissociation scan — optional, and the long one

Build, validate and run HI-VQE at every geometry from 1.70 Å out to 6.00 Å. This is
where the paper's claim actually lives: HI-VQE inside chemical accuracy of CASCI at
*every* bond length while Hartree–Fock is off by hundreds of millihartree at long
range. At a stretched bond the fragments are Li(²S) and LiS(²Π) — two open shells, no
single determinant — which is exactly why RHF fails there and exactly why the
benchmark is worth running.

**24 qubits at all 13 points.** The qubit count is CAS(12e,12o) — 12 spatial orbitals,
24 spin orbitals, one qubit each under Jordan–Wigner — and is a property of the active
space, not of the grid. The full determinant space is 853,776 at every geometry; only
the *subspace* HI-VQE selects out of it changes.

**Two settings differ from the single-point runs above, and both are about the
stretched end.**

`--energy-tolerance 1e-6 --patience 5` replaces the defaults of `1e-5` and `3`. The
long-bond points are where selected CI plateaus: near-degenerate configurations make
the energy crawl for several iterations before it drops again, and the default
patience calls that convergence and stops early — at 6.00 Å it stopped at 12,996
determinants with the 20,000 budget still unspent. The tighter pair spends the budget.

**Orbital continuation matters here.** RHF on a stretched bond has several solutions,
and a fresh atomic guess routinely lands on a different one, putting a step in the
curve that no correlation treatment can repair. Each point starts from the converged
orbitals of the **nearest geometry already built**, working outward from equilibrium.

### Read the two new columns

`S2` is ⟨S²⟩ of the HI-VQE state; `refS2` is ⟨S²⟩ of the CASCI reference. **0 is a
singlet, 2 a triplet.** They should agree at every point, and the geometry where both
switch from 0 to 2 is the singlet–triplet crossover — physics, not a glitch. If they
*disagree*, the error on that row is comparing two different spin states and is not a
number to quote.

This is worth watching because the reference solver was changed for exactly this
reason: PySCF's default for a closed-shell RHF reference is `direct_spin0`, which
cannot return a triplet at all, while HI-VQE searches the whole S_z=0 sector. At a
dissociated bond that mismatch made a correct run come back *below* the reference and
be reported `INCONSISTENT`. The reference is now `direct_spin1` — the same constraint
HI-VQE is under. See `_use_full_sz_solver` in `hamiltonian.py`.

Budget **15–25 min** for the scan, plus the `prepare` loop ahead of it. Adding
`--optimizer none --workers 2` roughly halves the scan at no measured cost in
accuracy. Two workers is right for a free Colab runtime — each holds its own copy of
the Hamiltonian, a few hundred MB at 24 qubits.

**Skip this cell entirely if you only want the single point above.** Everything
downstream still works; the dissociation figure is simply absent.

In [ ]:
import subprocess
from molecule import DISSOCIATION_DISTANCES, LI2S

# All 13 points of the published grid. Any subset works -- but keep a geometry
# either side of 2.10 or section 12 will decline to report a dissociation
# energy, because an unbracketed minimum has no well to measure.
SCAN_DISTANCES = DISSOCIATION_DISTANCES

ordered = sorted(SCAN_DISTANCES, key=lambda r: abs(r - LI2S.equilibrium_bond_angstrom))
prepared: list[float] = []
for distance in ordered:
    command = ["python", "run.py", "prepare", "--distance", f"{distance:.3f}"]
    if prepared:
        # Continue from the NEAREST geometry already converged, not merely the
        # previous one. Ordered outward from equilibrium the previous point can
        # be on the other side of the minimum, and seeding a stretched RHF from
        # a compressed one is how a curve picks up a step that no correlation
        # treatment can fix.
        source = min(prepared, key=lambda r: abs(r - distance))
        command += ["--continue-from", f"{source:.3f}"]
    print("=" * 70)
    print(" ".join(command))
    if subprocess.run(command).returncode == 0:
        prepared.append(distance)

scan_distances = " ".join(f"{distance:.3f}" for distance in SCAN_DISTANCES)
print("=" * 70)
print(f"prepared {len(prepared)}/{len(SCAN_DISTANCES)} geometries")
print("scanning:", scan_distances)

!python run.py validate --all
!python run.py scan --distances {scan_distances} --max-determinants 20000 --max-iterations 12 --energy-tolerance 1e-6 --patience 5


## 10. Classical baselines, and the measurement-cost claim recomputed

MP2, CCSD and CCSD(T) **frozen to the identical active space**, so the quantum error
has a scale. Running them on all 22 electrons would correlate orbitals CASCI never
touched and produce a number that is meaningless for comparison.

Watch the **T1 diagnostic**. Above ~0.02 the reference determinant no longer
dominates and CCSD(T) stops being a gold standard — which is precisely what a
breaking Li–S bond does, and precisely why CASCI is the reference here.

`paulis` recomputes the 15,697-Pauli-word cost of a conventional VQE from the cached
integrals, so the headline comparison is a number you check rather than one you
quote.

In [ ]:
!python run.py classical --all
!python run.py paulis --distance 2.10

## 11. The figures

Seven figures, one question each. None of them are decoration, and the palette is
consistent across all of them: the CASCI reference is always black, Hartree–Fock
always grey, HI-VQE always the strong blue, the classical post-HF methods the warm
end, and the 1.6 mHa chemical-accuracy line always the same green dash. A reader who
learns the colours on one figure keeps them.

| # | Figure | The question it answers |
|---|---|---|
| 1 | `dissociation` | Does HI-VQE track CASCI **everywhere**, not just near the minimum? Total energies on top, error against CASCI on a log axis below, with HF, MP2, CCSD and CCSD(T) on the same axes. *(needs §9)* |
| 2 | `convergence` | Did it converge, and to what — error per iteration, beside the subspace dimension against the full 853,776. |
| 3 | `energy` | Where did the energy actually **go**? The same run in absolute hartree, between the Hartree–Fock and CASCI lines, so the correlation energy being recovered is a visible distance rather than a number to be trusted. One geometry only — total energies at different bond lengths differ by far more than the correlation energy, and mixing them flattens the plot. |
| 4 | `compression` | How much accuracy did each determinant buy? Error against subspace size, log–log, with the full CAS marked. |
| 5 | `ablation` | What did the quantum sampler contribute over the classical-only control? Blue is quantum, orange is classical-only. |
| 6 | `occupancies` | Where the electrons sit. Deviation from 2 and 0 **is** the multireference character, and watching it grow from equilibrium to dissociation is the physics. *(second bar needs §9)* |
| 7 | `cost` | 15,697 measurement settings per iteration versus 1, drawn to scale, beside the sampling circuit's own gate counts. |

Figures render inline here; §12 writes the identical charts as PNGs. If one is
missing, it is because the run it needs is missing — the cell says which.

In [ ]:
%matplotlib inline
import importlib

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

import visualize

# Re-import so an edit to visualize.py takes effect without restarting Colab.
importlib.reload(visualize)

QUESTION = {
    "dissociation": "Does it track CASCI everywhere, not just near the minimum?",
    "convergence":  "Did it converge, and how much of the space did it hold?",
    "energy":       "Where did the energy go, in hartree, between HF and CASCI?",
    "compression":  "How much accuracy did each determinant buy?",
    "ablation":     "What did the quantum sampler contribute over classical alone?",
    "occupancies":  "Where do the electrons sit, and how multireference is that?",
    "cost":         "15,697 measurement settings per iteration, versus 1.",
}
ORDER = ["dissociation", "convergence", "energy", "compression", "ablation",
         "occupancies", "cost"]

figures = visualize.build_figures("results")
if not figures:
    print("Nothing in results/ yet -- run section 7 first.")
else:
    ordered = [name for name in ORDER if name in figures]
    ordered += [name for name in figures if name not in ORDER]
    for name in ordered:
        display(Markdown(f"#### {name} \u2014 *{QUESTION.get(name, '')}*"))
        display(figures[name])
    absent = [name for name in ORDER if name not in figures]
    if absent:
        print(f"Not drawn, because the runs they need are missing: {', '.join(absent)}")
plt.close("all")

And the same runs as a table, sorted by error, so the figures have numbers behind
them. `subspace_fraction` is the column to read next to the error: it is the paper's
compression claim, measured. The `+PT2 (mHa)` column is the perturbatively corrected
error — `--` when the correction was not computed, and suffixed with `!` when it was
computed but fell outside the domain where perturbation theory means anything.

In [ ]:
!python run.py summary --sort error

## 12. The assembled report

`plot` writes every figure to `results/figures/` as a PNG. `report` assembles the
runs, the ablation table, the provenance hashes and the figures into `report.md` and
a **standalone `report.html`** — the figures are embedded as data URIs, so that one
file is the whole report and survives being emailed.

The report also computes the **dissociation energy** from the curve and its error
against CASCI. That is the number to quote: an energy *difference* cancels most of
the systematic error an absolute total energy carries, so it is the stricter test of
consistency.

In [ ]:
!python run.py plot
!python run.py report

from IPython.display import HTML

HTML(open("results/report.html", encoding="utf-8").read())

## 13. Download everything

Two zips. **The cache is the expensive one** — it is what PySCF spent its time on,
and with it every analysis here re-runs on a laptop, including a Windows one with no
PySCF at all. It carries its own specification hash and the receipts are bound to it,
so a stale or mismatched file is a hard error rather than a wrong answer. Copying it
between machines is safe by construction.

`results.zip` carries every run's JSON, the figures as PNGs, and both reports.

On the Windows machine: unzip both next to `run.py`, and `validate`, `hivqe`, `scan`,
`summary`, `plot` and `report` all work natively from there on.

In [ ]:
import shutil
from pathlib import Path

for name in ("results", "cache"):
    if not Path(name).is_dir():
        print(f"{name}/ does not exist -- nothing to package")
        continue
    archive = shutil.make_archive(f"/content/li2s-24q_{name}", "zip", ".", name)
    print(f"{archive}  ({Path(archive).stat().st_size / 1e6:.2f} MB)")

try:
    from google.colab import files
    for name in ("results", "cache"):
        path = Path(f"/content/li2s-24q_{name}.zip")
        if path.is_file():
            files.download(str(path))
except ImportError:
    print("\nNot on Colab -- the zips are next to this notebook.")